In [0]:
%run ../init/kafka_config

In [0]:
from pyspark.sql.functions import col, from_json, when, round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
order_schema = StructType([
    StructField("order_id",      StringType(),  nullable=False),
    StructField("user_id",       StringType(),  nullable=False),
    StructField("item_id",       StringType(),  nullable=False),
    StructField("item_quantity", IntegerType(), nullable=False),
    StructField("category",      StringType(),  nullable=False)
])


In [0]:
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
    .option("subscribe", "orders") \
    .option("startingOffsets", "earliest") \
    .load()

In [0]:
df_orders = df_raw \
    .selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), order_schema).alias("data")) \
    .select("data.*")

In [0]:
users_static     = spark.read.table("users")
items_static     = spark.read.table("items")
inventory_static = spark.read.table("inventory")

In [0]:
df_enriched = df_orders \
    .join(users_static,     df_orders.user_id == users_static.user_id,         "left") \
    .join(items_static,     df_orders.item_id == items_static.item_id,         "left") \
    .join(inventory_static, df_orders.item_id == inventory_static.item_id,     "left") \
    .select(
        df_orders.order_id,
        df_orders.user_id,
        df_orders.item_id,
        df_orders.item_quantity,
        df_orders.category,
        users_static.username,
        users_static.balance,
        items_static.item_name,
        items_static.price,
        inventory_static.quantity_available
    )

In [0]:
df_validated = df_enriched \
    .withColumn("total_cost",
        col("price") * col("item_quantity")
    ) \
    .withColumn("status",
        when(col("username").isNull(),                                    "FAILED_INVALID_USER")
        .when(col("item_name").isNull(),                                  "FAILED_INVALID_ITEM")
        .when(col("quantity_available") < col("item_quantity"),           "FAILED_INVENTORY")
        .when(col("balance") < col("total_cost"),                         "FAILED_BALANCE")
        .otherwise(                                                        "ORDER_PLACED")
    ) \
    .withColumn("balance_after",
        when(col("status") == "ORDER_PLACED",
            round(col("balance") - col("total_cost"), 2)
        ).otherwise(col("balance"))
    ) \
    .withColumn("quantity_after",
        when(col("status") == "ORDER_PLACED",
            col("quantity_available") - col("item_quantity")
        ).otherwise(col("quantity_available"))
    )

In [0]:
checkpoint_base = "/Workspace/Users/hemasundher0000@gmail.com/Confluent_Kafka_Project/checkpoints"

In [0]:
def process_batch(batch_df, batch_id):

    from pyspark.sql.functions import current_timestamp, to_json, struct, when, col

    # ── Duplicate check ────────────────────
    existing_orders = spark.read.table("orders").select("order_id")

    new_orders_df = batch_df.join(
        existing_orders,
        on="order_id",
        how="left_anti"
    ).select("order_id","user_id","item_id","item_quantity","category")

    if new_orders_df.count() == 0:
        return

    # ── Re-read static tables fresh ────────
    users_df = spark.read.table("users") \
        .select(
            col("user_id"),
            col("username"),
            col("balance")
        )

    items_df = spark.read.table("items") \
        .select(
            col("item_id").alias("item_id_items"),
            col("item_name"),
            col("price")
        )

    inventory_df = spark.read.table("inventory") \
        .select(
            col("item_id").alias("item_id_inv"),
            col("quantity_available")
        )

    # ── Enrich and validate ────────────────
    enriched_df = new_orders_df \
        .join(users_df,
            on="user_id",
            how="left") \
        .join(items_df,
            new_orders_df.item_id == items_df.item_id_items,
            how="left") \
        .join(inventory_df,
            new_orders_df.item_id == inventory_df.item_id_inv,
            how="left") \
        .drop("item_id_items", "item_id_inv") \
        .withColumn("total_cost",
            col("price") * col("item_quantity")
        ) \
        .withColumn("status",
            when(col("username").isNull(),                          "FAILED_INVALID_USER")
            .when(col("item_name").isNull(),                        "FAILED_INVALID_ITEM")
            .when(col("quantity_available") < col("item_quantity"), "FAILED_INVENTORY")
            .when(col("balance") < col("total_cost"),               "FAILED_BALANCE")
            .otherwise(                                              "ORDER_PLACED")
        ) \
        .withColumn("balance_after",
            when(col("status") == "ORDER_PLACED",
                col("balance") - col("total_cost")
            ).otherwise(col("balance"))
        )

    # ── Materialize to temp view ───────────
    enriched_df.createOrReplaceTempView("temp_batch_view")
    materialized_df = spark.table("temp_batch_view")

    # ── Split success and failed ───────────
    success_df = materialized_df.filter(col("status") == "ORDER_PLACED")
    failed_df  = materialized_df.filter(col("status") != "ORDER_PLACED")

    # ── 1. Write balance_events ────────────
    success_df \
        .withColumn("amount_change", -col("total_cost")) \
        .withColumn("timestamp", current_timestamp()) \
        .select("user_id","amount_change","order_id","timestamp") \
        .write.format("delta").mode("append").saveAsTable("balance_events")

    # ── 2. Write inventory_events ──────────
    success_df \
        .withColumn("quantity_change", -col("item_quantity")) \
        .withColumn("timestamp", current_timestamp()) \
        .select("item_id","quantity_change","order_id","timestamp") \
        .write.format("delta").mode("append").saveAsTable("inventory_events")

    # ── 3. Write notifications ─────────────
    success_df \
        .withColumn("timestamp", current_timestamp()) \
        .select("user_id","username","item_name","item_quantity","balance_after","timestamp") \
        .write.format("delta").mode("append").saveAsTable("notifications")

    # ── 4. Write order_logs (success) ──────
    success_df \
        .withColumn("timestamp", current_timestamp()) \
        .select(
            col("order_id"),
            col("user_id"),
            col("item_id"),
            col("item_quantity"),
            col("status"),
            col("balance").alias("user_balance_before"),
            col("timestamp")
        ) \
        .write.format("delta").mode("append").saveAsTable("order_logs")

    # ── 5. Write order_logs (failed) ───────
    failed_df \
        .withColumn("timestamp", current_timestamp()) \
        .withColumn("user_balance_before",
            when(col("balance").isNull(), 0.0)
            .otherwise(col("balance"))
        ) \
        .select("order_id","user_id","item_id","item_quantity","status","user_balance_before","timestamp") \
        .write.format("delta").mode("append").saveAsTable("order_logs")

    # ── 6. Write to dead_letter topic ──────
    failed_df \
        .select(
            to_json(struct(
                col("order_id"),
                col("user_id"),
                col("item_id"),
                col("item_quantity"),
                col("category"),
                col("status")
            )).alias("value")
        ) \
        .write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", bootstrap) \
        .option("kafka.security.protocol", "SASL_SSL") \
        .option("kafka.sasl.mechanism", "PLAIN") \
        .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
        .option("topic", "dead_letter") \
        .save()

    # ── 7. Write orders LAST ───────────────
    success_df \
        .withColumn("timestamp", current_timestamp()) \
        .select("order_id","user_id","item_id","item_quantity","category","status","timestamp") \
        .write.format("delta").mode("append").saveAsTable("orders")

In [0]:
query = df_validated.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", f"{checkpoint_base}/stream_order_processor") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()
print("\n✓ Stream processing complete")


In [0]:
print("── Orders ──────────────────────────────")
spark.read.table("orders").show(truncate=False)

print("── Order Logs ──────────────────────────")
spark.read.table("order_logs").show(truncate=False)

print("── Balance Events ──────────────────────")
spark.read.table("balance_events").show(truncate=False)

print("── Inventory Events ────────────────────")
spark.read.table("inventory_events").show(truncate=False)

print("── Notifications ───────────────────────")
spark.read.table("notifications").show(truncate=False)

In [0]:
# from delta.tables import DeltaTable
# from pyspark.sql.functions import col, lit

# DeltaTable.forName(spark, "orders").delete()
# DeltaTable.forName(spark, "order_logs").delete()
# DeltaTable.forName(spark, "balance_events").delete()
# DeltaTable.forName(spark, "inventory_events").delete()
# DeltaTable.forName(spark, "notifications").delete()

# for user_id, balance in [("U001",1500.0),("U002",800.0),("U003",300.0),("U004",2000.0),("U005",150.0)]:
#     DeltaTable.forName(spark, "users").update(
#         condition = col("user_id") == user_id,
#         set       = {"balance": lit(balance)}
#     )

# for item_id, qty in [("I001",10),("I002",25),("I003",15),("I004",50),("I005",100)]:
#     DeltaTable.forName(spark, "inventory").update(
#         condition = col("item_id") == item_id,
#         set       = {"quantity_available": lit(qty)}
#     )

# dbutils.fs.rm(
#     "/Workspace/Users/hemasundher0000@gmail.com/Confluent_Kafka_Project/checkpoints/stream_order_processor",
#     recurse=True
# )
# print("✓ Reset complete")